<a href="https://colab.research.google.com/github/wallynovak/phosphatase/blob/main/Acid_Phosphatase_Analysis_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img align="right" src="https://github.com/wallynovak/nematode_aggregate_analysis/raw/main/images/m2mlogo.png" width="150" height="150" />

# Molecules to Medicine Summer Camp
## In Vitro Drug Screening and Enzyme Activity
### Acid Phosphatase Data Analysis

**Purpose:**
In this notebook you will analyze data from two experiments:

- **Part 1:** You screened five household compounds for their ability
  to inhibit wheat germ acid phosphatase. You will calculate the
  activity in each well and visualize the results.

- **Parts 2–4:** You measured enzyme activity at six substrate
  concentrations, with no inhibitor and with two concentrations
  of phosphate as a competitive inhibitor. You will determine the
  kinetic parameters Km and Vmax by two methods — non-linear
  curve fitting and a Lineweaver-Burk double-reciprocal plot —
  and use these to determine what type of inhibitor phosphate is.

**In this notebook you will learn to:**
- Use Python to convert raw absorbance readings to enzyme activity
- Plot and interpret a Michaelis-Menten curve
- Fit the Michaelis-Menten equation to experimental data
- Construct and interpret a Lineweaver-Burk plot
- Compare kinetic parameters across inhibitor conditions

**Note on pathlength:**
Your plate reader has automatically corrected all absorbance readings
to a **1 cm pathlength** using its PathCheck® function.
All data entry cells use a pathlength of **1.0 cm**.

**Status:** Ready for the Molecules to Medicine Summer Camp — 2026

**License**

<img src="https://github.com/codeBMB/April26_Rutgers-RCSB/raw/main/images/by-nc-sa.png" width="100"/>
CC BY-NC-SA — free to use and adapt for non-commercial educational purposes with attribution.

---
**Authorship:** Wally Novak

**Acknowledgements:** Supported by NSF IUSE 2518733.

**Contact:** novakw [at] wabash [dot] edu

# 0. How to Run This Notebook

To run a cell, click the **▶ play button** on the left side of the cell.

A cell is still running if you see a stop button with a moving circle.
A completed cell shows a number in brackets (e.g. [1]) and a checkmark.

**Always run the cells in order from top to bottom.**

> ⚠️ If you restart the notebook at any point, run all cells again
> from the beginning — Python forgets everything when it restarts.

# 1. Scientific Background

## Phosphatases and Disease

**Phosphatases** are enzymes that remove a phosphate group (PO₄³⁻)
from another molecule by cleaving a phosphoester bond using water:

$$Substrate\text{-}OPO_3^{2-} + H_2O \rightarrow Substrate\text{-}OH + PO_4^{3-}$$

Two human phosphatases — **Protein Phosphatase 2A (PP2A)** and
**Protein Tyrosine Phosphatase 1B (PTP1B)** — are involved in
Alzheimer's disease and are active targets of new drug development.
In this lab you work with **wheat germ acid phosphatase**, which is
easier and safer to handle in a teaching lab and behaves similarly.

---

## The pNPP Assay

We cannot watch phosphate being released directly, so we use a
**colorimetric proxy substrate**: **p-nitrophenyl phosphate (pNPP)**.

The enzyme cleaves pNPP to produce **p-nitrophenol (pNP)**.
At high pH (after adding NaOH to stop the reaction), pNP is
ionized to **p-nitrophenoxide**, which is bright yellow and
absorbs light at **405 nm**.

> **More enzyme activity → more pNP produced → deeper yellow color
> → higher absorbance at 405 nm**

---

## Beer-Lambert Law

The absorbance reading from the plate reader is related to
concentration by the **Beer-Lambert Law**:

$$A = \varepsilon \cdot l \cdot c$$

where:
- $A$ = absorbance (measured by the plate reader)
- $\varepsilon$ = molar extinction coefficient of pNP = **18,000 M⁻¹ cm⁻¹**
- $l$ = pathlength (corrected to **1.0 cm** by the plate reader)
- $c$ = concentration of pNP produced

Rearranging to get the amount of product formed, then dividing by
time gives us **enzyme activity** in Units per minute (U/min),
where **1 U = 1 nmol of pNPP hydrolyzed per minute**.

---

## The Inhibitor: Inorganic Phosphate

In Parts 2–4, we use **inorganic phosphate (Pi)** as our inhibitor.
Phosphate is the natural product of the reaction and competes with
pNPP for the enzyme's active site — it is a **competitive inhibitor**.

Studying how phosphate affects the kinetics of acid phosphatase
is directly relevant to drug development: understanding how
molecules block phosphatase active sites helps guide the design
of drugs that could slow Alzheimer's disease progression.

---

## Michaelis-Menten Kinetics

The **Michaelis-Menten equation** describes how reaction velocity
(v) depends on substrate concentration ([S]):

$$v = \frac{V_{max} \cdot [S]}{K_m + [S]}$$

| Parameter | Symbol | Meaning |
|---|---|---|
| Maximum velocity | $V_{max}$ | Velocity when all enzyme active sites are saturated |
| Michaelis constant | $K_m$ | Substrate concentration at half $V_{max}$ |

**Competitive inhibitors** increase the apparent Km but do not
change Vmax (at saturating substrate, pNPP outcompetes Pi).

---

## Lineweaver-Burk Plot

Taking the reciprocal of both sides of the MM equation gives the
**Lineweaver-Burk** (double-reciprocal) equation:

$$\frac{1}{v} = \frac{K_m}{V_{max}} \cdot \frac{1}{[S]} + \frac{1}{V_{max}}$$

Plotting **1/v** vs **1/[S]** gives a straight line where:
- **Y-intercept** = $1/V_{max}$
- **Slope** = $K_m / V_{max}$
- **X-intercept** = $-1/K_m$

For a **competitive inhibitor**, the lines for different inhibitor
concentrations will **intersect on the y-axis**
(same Vmax, different slopes).

# 2. Setup

Run the cell below to load all the Python tools we need.
This cell must be run first — everything else depends on it.

| Library | Purpose |
|---|---|
| `numpy` | Numerical calculations and arrays |
| `matplotlib` | Plotting graphs |
| `scipy.optimize` | Non-linear curve fitting for Michaelis-Menten |
| `scipy.stats` | Linear regression for Lineweaver-Burk |

In [ ]:
# Run this cell first — loads all required tools
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from scipy.stats import linregress

# ── Define the Michaelis-Menten equation once (used throughout) ──
def michaelis_menten(S, Vmax, Km):
    """
    Michaelis-Menten equation.
    S    : substrate concentration (µM)
    Vmax : maximum reaction velocity (U/min)
    Km   : Michaelis constant (µM)
    """
    return (Vmax * S) / (Km + S)

# ── Shared assay constants ────────────────────────────────────────
ASSAY_TIME     = 5        # reaction time in minutes
MOL_EXTINCTION = 18000    # molar extinction coefficient of pNP, M⁻¹ cm⁻¹

# Total volume after adding NaOH:
#   200 µL reaction + 100 µL NaOH stop solution = 300 µL = 300e-6 L
# We use this in the Beer-Lambert calculation below.
TOTAL_VOL_L    = 300e-6   # total volume in the well after NaOH addition (L)

print("✅ Setup complete. All tools loaded.")
print(f"   Assay time:            {ASSAY_TIME} min")
print(f"   Molar extinction (ε):  {MOL_EXTINCTION} M⁻¹ cm⁻¹")
print(f"   Total well volume:     {TOTAL_VOL_L*1e6:.0f} µL")
print()
print("Ready to enter your data below.")

# Part 1 — Library Screening

## Entering Your Data

Replace the example absorbance values below with your own
plate reader readings at **405 nm** from Column 1.

Each row in the list is formatted as:
`(Row label, Absorbance at 405 nm, Pathlength in cm)`

Your plate reader has corrected the pathlength to **1.0 cm**,
so all pathlength values should remain **1.0**.

| Row | Well Contents |
|---|---|
| A | Blank — buffer + pNPP, **no enzyme** (background control) |
| B | Uninhibited control — enzyme + pNPP, no inhibitor |
| C–G | Enzyme + pNPP + one test compound each |

> **Note on the blank:**
> Your plate reader may already have zeroed the absorbance
> against a blank well. The code below subtracts Row A
> explicitly as a safeguard — if the reader already zeroed
> it, Row A will be ≈ 0 and the subtraction will have no effect.

In [ ]:
# ── Part 1: Enter your absorbance data from Column 1 ─────────
# Format: (Row label, Absorbance at 405 nm, Pathlength in cm)
# Replace the example absorbance values with your own readings.
# Keep pathlength as 1.0 — the plate reader has already corrected this.

abs405_part1 = [
    ("A", 0.10, 1.0),   # Blank (no enzyme)        ← replace 0.10
    ("B", 0.57, 1.0),   # No inhibitor (control)   ← replace 0.57
    ("C", 0.55, 1.0),   # Test compound 1           ← replace 0.55
    ("D", 0.26, 1.0),   # Test compound 2           ← replace 0.26
    ("E", 0.12, 1.0),   # Test compound 3           ← replace 0.12
    ("F", 0.57, 1.0),   # Test compound 4           ← replace 0.57
    ("G", 0.49, 1.0),   # Test compound 5           ← replace 0.49
]

# ── Blank subtraction ─────────────────────────────────────────
# Row A contains no enzyme — any absorbance it shows is background.
# We subtract this from all other rows before calculating activity.
# If your plate reader already zeroed against a blank, Row A ≈ 0
# and this subtraction will have no effect.
blank_abs = abs405_part1[0][1]
print(f"Row A (blank) absorbance: {blank_abs:.3f}")
print(f"This value will be subtracted from all other rows.")
print()

# ── Calculate enzyme activity ─────────────────────────────────
# Activity (U/min) = (A_corrected × V_total × 1e9) / (ε × l × t)
#
# Where:
#   A_corrected = absorbance after blank subtraction
#   V_total     = 300e-6 L  (200 µL reaction + 100 µL NaOH stop)
#   1e9         = converts mol → nmol  (1 U = 1 nmol/min)
#   ε           = 18,000 M⁻¹ cm⁻¹    (molar extinction of pNP)
#   l           = pathlength in cm (1.0, corrected by plate reader)
#   t           = assay time in minutes (5)

activity1 = []
print(f"{'Row':<6} {'Abs (raw)':<12} {'Abs (corrected)':<18} {'Activity (U/min)'}")
print("-" * 58)

for row_label, abs_raw, pathlength in abs405_part1:
    abs_corrected = abs_raw - blank_abs
    if row_label == "A":
        abs_corrected = 0.0
    act = (abs_corrected * TOTAL_VOL_L * 1e9
           / (MOL_EXTINCTION * pathlength * ASSAY_TIME))
    act = round(act, 3)
    activity1.append((row_label, act))
    print(f"{row_label:<6} {abs_raw:<12.3f} {abs_corrected:<18.3f} {act}")

## Part 1 Results — Bar Chart

Run the cell below to generate a bar chart of your results.

- **Row A** (blank) should be near zero
- **Row B** (no inhibitor) is your positive control — the maximum activity
- **Rows C–G** show whether each compound inhibited the enzyme

> A row with **lower activity than Row B** suggests inhibition.
> A row with **activity near zero** is a strong inhibitor candidate.

💡 **Discuss with your group:** Which compounds inhibited the enzyme?

In [ ]:
# Bar chart of Part 1 results
rows       = [item[0] for item in activity1]
activities = [item[1] for item in activity1]

bar_colors = ['steelblue' if r != 'B' else 'darkorange' for r in rows]

plt.figure(figsize=(9, 5))
bars = plt.bar(rows, activities, color=bar_colors, edgecolor='white', linewidth=0.8)

for bar, val in zip(bars, activities):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{val:.2f}', ha='center', va='bottom', fontsize=10)

control_activity = activity1[1][1]
plt.axhline(y=control_activity, color='darkorange', linestyle='--', alpha=0.6,
            label=f'Uninhibited control (Row B = {control_activity:.3f} U/min)')

plt.xlabel('Well (Row)', fontsize=12)
plt.ylabel('Enzyme Activity (U/min)', fontsize=12)
plt.title('Part 1 — Effect of Household Compounds on Acid Phosphatase Activity', fontsize=13)
plt.legend(fontsize=10)
plt.ylim(bottom=0)
plt.grid(axis='y', alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("Inhibition relative to uninhibited control (Row B):")
for row_label, act in activity1:
    if row_label == 'A':
        continue
    pct = max(0, (1 - act / control_activity) * 100) if control_activity > 0 else 0
    flag = " ← potential inhibitor" if pct > 20 else ""
    print(f"  Row {row_label}: {act:.3f} U/min  ({pct:.1f}% inhibition){flag}")

## Part 1 Discussion

Based on your bar chart, answer the following:

1. Which of your test compounds inhibited acid phosphatase?
   (Look for rows with significantly lower activity than Row B.)

2. Is the inhibition complete (activity near zero) or partial?

3. Could the inhibition be due to a non-specific effect such as
   changing the pH? How might you test this?

---

**We will discuss results as a class before moving on to Part 2.**

# Part 2 — Enzyme Kinetics: No Inhibitor

## Background

In Part 2 you measured enzyme activity at **six different
substrate concentrations** (0–600 µM pNPP) with no inhibitor.
This gives us the baseline Michaelis-Menten kinetics of
acid phosphatase.

## Entering Your Data

Replace the example absorbance values with your readings
from **Column 2** of your plate.

Each row is formatted as:
`(Row label, Absorbance at 405 nm, Pathlength in cm, [pNPP] in µM)`

Your plate reader has corrected the pathlength to **1.0 cm**.
Do **not** change the [pNPP] concentration values.

| Row | [pNPP] µM | Expected behavior |
|---|---|---|
| A | 0 | Near zero — no substrate, no product |
| B | 50 | Low activity |
| C | 100 | Rising activity |
| D | 200 | Continuing to rise |
| E | 400 | Approaching plateau |
| F | 600 | Near maximum (approaching Vmax) |

In [ ]:
# ── Part 2: No inhibitor — enter your Column 2 data ──────────
# Format: (Row label, Absorbance at 405 nm, Pathlength in cm, [pNPP] in µM)
# Replace ONLY the absorbance values (column 2).
# Pathlength stays at 1.0. Do NOT change the [pNPP] values.

abs405_part2 = [
    ("A", 0.00, 1.0,   0),   # 0 µM pNPP     ← replace absorbance
    ("B", 0.13, 1.0,  50),   # 50 µM pNPP    ← replace absorbance
    ("C", 0.18, 1.0, 100),   # 100 µM pNPP   ← replace absorbance
    ("D", 0.25, 1.0, 200),   # 200 µM pNPP   ← replace absorbance
    ("E", 0.29, 1.0, 400),   # 400 µM pNPP   ← replace absorbance
    ("F", 0.30, 1.0, 600),   # 600 µM pNPP   ← replace absorbance
]

activity2 = []
print("Part 2 — No Inhibitor")
print(f"{'[pNPP] µM':<14} {'Absorbance (AU)':<18} {'Activity (U/min)'}")
print("-" * 50)

for row_label, abs_val, pathlength, conc_pNPP in abs405_part2:
    act = (abs_val * TOTAL_VOL_L * 1e9
           / (MOL_EXTINCTION * pathlength * ASSAY_TIME))
    act = round(act, 4)
    activity2.append((conc_pNPP, act))
    print(f"{conc_pNPP:<14} {abs_val:<18.3f} {act}")

S_data2 = np.array([d[0] for d in activity2])
V_data2 = np.array([d[1] for d in activity2])

## Part 2 — Michaelis-Menten Scatter Plot

Run the cell below to plot your data.

You should see velocity rising quickly at low substrate
concentrations and then leveling off toward a plateau (Vmax).

**Estimating Vmax and Km visually:**
1. **Vmax** ≈ the plateau value your curve is approaching
2. **Km** ≈ the substrate concentration at **½ × Vmax**

Write down your estimates — you will enter them in the
next cell as starting guesses for the curve fitting.

In [ ]:
# Michaelis-Menten scatter plot — Part 2 (no inhibitor)
plt.figure(figsize=(8, 5))
plt.scatter(S_data2, V_data2, color='steelblue', s=80, zorder=5,
            label='No inhibitor')

max_v = max(V_data2)
plt.axhline(y=max_v,   linestyle=':', color='gray',      alpha=0.6,
            label=f'Approx. Vmax ≈ {max_v:.3f} U/min')
plt.axhline(y=max_v/2, linestyle=':', color='lightgray', alpha=0.6,
            label=f'½ Vmax ≈ {max_v/2:.3f} U/min')

plt.xlabel('[pNPP] (µM)', fontsize=12)
plt.ylabel('Activity (U/min)', fontsize=12)
plt.title('Part 2 — Michaelis-Menten Plot: No Inhibitor', fontsize=13)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Maximum observed activity: {max_v:.4f} U/min")
print(f"Use this to guide your Vmax estimate in the next cell.")

## Part 2 — Curve Fitting

Before running the next cell, replace:
- `vmax_init` with your visual estimate of **Vmax**
- `km_init` with your visual estimate of **Km**

These starting guesses help the fitting algorithm converge.
A good starting guess for `vmax_init` is the highest activity
you observed. A good starting guess for `km_init` is the
substrate concentration where activity was about half the maximum.

In [ ]:
# ── Part 2 curve fitting ──────────────────────────────────────
# 🔧 Replace these with your visual estimates from the plot above
vmax_init = 1.0    # 🔧 your estimate of Vmax (U/min)
km_init   = 100.0  # 🔧 your estimate of Km (µM)

params2, covariance2 = curve_fit(
    michaelis_menten, S_data2, V_data2,
    p0=[vmax_init, km_init],
    bounds=(0, np.inf)
)
Vmax_fit2, Km_fit2 = params2

# R² — how well does the fitted curve match your data?
V_predicted2 = michaelis_menten(S_data2, Vmax_fit2, Km_fit2)
ss_res2 = np.sum((V_data2 - V_predicted2)**2)
ss_tot2 = np.sum((V_data2 - np.mean(V_data2))**2)
R2_mm2  = 1 - (ss_res2 / ss_tot2) if ss_tot2 > 0 else 0

print("Part 2 — No Inhibitor: Michaelis-Menten Curve Fit")
print("=" * 50)
print(f"  Fitted Vmax:  {Vmax_fit2:.4f} U/min")
print(f"  Fitted Km:    {Km_fit2:.4f} µM")
print(f"  R²:           {R2_mm2:.4f}")
print(f"  Fit quality:  {'Excellent ✅' if R2_mm2>=0.99 else 'Acceptable ⚠️' if R2_mm2>=0.95 else 'Poor — check data ❌'}")

# Plot data with fitted curve
S_fit2 = np.linspace(0, max(S_data2) * 1.2, 200)
V_fit2 = michaelis_menten(S_fit2, Vmax_fit2, Km_fit2)

plt.figure(figsize=(8, 5))
plt.scatter(S_data2, V_data2, color='steelblue', s=80, zorder=5, label='Experimental data')
plt.plot(S_fit2, V_fit2, color='red', linewidth=2,
         label=f'MM fit: Vmax={Vmax_fit2:.3f} U/min, Km={Km_fit2:.1f} µM')
plt.axhline(y=Vmax_fit2,   color='gray',      linestyle=':', alpha=0.7,
            label=f'Vmax = {Vmax_fit2:.3f}')
plt.axhline(y=Vmax_fit2/2, color='lightgray', linestyle=':', alpha=0.7)
plt.axvline(x=Km_fit2,     color='lightgray', linestyle=':', alpha=0.7,
            label=f'Km = {Km_fit2:.1f} µM')
plt.xlabel('[pNPP] (µM)', fontsize=12)
plt.ylabel('Activity (U/min)', fontsize=12)
plt.title('Part 2 — Michaelis-Menten Curve Fit: No Inhibitor', fontsize=13)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Part 3 — Enzyme Kinetics: Low Phosphate Concentration

## Background

In Part 3 you used the same substrate concentration range but
added **10 µL of phosphate solution** to each well.

**Inorganic phosphate (Pi)** is a competitive inhibitor of
acid phosphatase — it competes with pNPP for the active site
because it resembles the phosphate group that the enzyme
normally cleaves.

If phosphate is a **competitive inhibitor**, you should observe:
- A higher apparent Km (more substrate needed to reach half Vmax)
- Approximately the same Vmax

## Entering Your Data

Replace the example absorbance values with your readings from
**Column 3** of your plate.

Each row is formatted as:
`(Row label, Absorbance at 405 nm, Pathlength in cm, [pNPP] in µM)`

Do **not** change the [pNPP] concentration values.

In [ ]:
# ── Part 3: Low phosphate (10 µL) — enter your Column 3 data ─
# Format: (Row label, Absorbance at 405 nm, Pathlength in cm, [pNPP] in µM)
# Replace ONLY the absorbance values.

abs405_part3 = [
    ("A", 0.00, 1.0,   0),   # 0 µM pNPP     ← replace absorbance
    ("B", 0.08, 1.0,  50),   # 50 µM pNPP    ← replace absorbance
    ("C", 0.12, 1.0, 100),   # 100 µM pNPP   ← replace absorbance
    ("D", 0.15, 1.0, 200),   # 200 µM pNPP   ← replace absorbance
    ("E", 0.19, 1.0, 400),   # 400 µM pNPP   ← replace absorbance
    ("F", 0.20, 1.0, 600),   # 600 µM pNPP   ← replace absorbance
]

activity3 = []
print("Part 3 — Low Phosphate Inhibitor (10 µL)")
print(f"{'[pNPP] µM':<14} {'Absorbance (AU)':<18} {'Activity (U/min)'}")
print("-" * 50)

for row_label, abs_val, pathlength, conc_pNPP in abs405_part3:
    act = (abs_val * TOTAL_VOL_L * 1e9
           / (MOL_EXTINCTION * pathlength * ASSAY_TIME))
    act = round(act, 4)
    activity3.append((conc_pNPP, act))
    print(f"{conc_pNPP:<14} {abs_val:<18.3f} {act}")

S_data3 = np.array([d[0] for d in activity3])
V_data3 = np.array([d[1] for d in activity3])

# Scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(S_data3, V_data3, color='royalblue', s=80, zorder=5)
plt.xlabel('[pNPP] (µM)', fontsize=12)
plt.ylabel('Activity (U/min)', fontsize=12)
plt.title('Part 3 — Michaelis-Menten Plot: Low Phosphate (10 µL)', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── 🔧 Update initial guesses for Part 3 ──────────────────────
vmax_init3 = 1.0    # 🔧 try your Part 2 Vmax as a starting point
km_init3   = 150.0  # 🔧 may be larger than Part 2 Km

params3, covariance3 = curve_fit(
    michaelis_menten, S_data3, V_data3,
    p0=[vmax_init3, km_init3],
    bounds=(0, np.inf)
)
Vmax_fit3, Km_fit3 = params3

V_predicted3 = michaelis_menten(S_data3, Vmax_fit3, Km_fit3)
ss_res3 = np.sum((V_data3 - V_predicted3)**2)
ss_tot3 = np.sum((V_data3 - np.mean(V_data3))**2)
R2_mm3  = 1 - (ss_res3 / ss_tot3) if ss_tot3 > 0 else 0

print("Part 3 — Low Phosphate: Michaelis-Menten Curve Fit")
print("=" * 50)
print(f"  Fitted Vmax:  {Vmax_fit3:.4f} U/min")
print(f"  Fitted Km:    {Km_fit3:.4f} µM")
print(f"  R²:           {R2_mm3:.4f}")
print(f"  Fit quality:  {'Excellent ✅' if R2_mm3>=0.99 else 'Acceptable ⚠️' if R2_mm3>=0.95 else 'Poor — check data ❌'}")

S_fit3 = np.linspace(0, max(S_data3) * 1.2, 200)
V_fit3 = michaelis_menten(S_fit3, Vmax_fit3, Km_fit3)

plt.figure(figsize=(8, 5))
plt.scatter(S_data3, V_data3, color='royalblue', s=80, zorder=5,
            label='Low phosphate data')
plt.plot(S_fit3, V_fit3, color='royalblue', linewidth=2,
         label=f'MM fit: Vmax={Vmax_fit3:.3f}, Km={Km_fit3:.1f} µM')
plt.xlabel('[pNPP] (µM)', fontsize=12)
plt.ylabel('Activity (U/min)', fontsize=12)
plt.title('Part 3 — MM Curve Fit: Low Phosphate (10 µL)', fontsize=13)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Part 4 — Enzyme Kinetics: High Phosphate Concentration

## Background

In Part 4 you doubled the phosphate concentration to **20 µL**
per well. This allows us to see whether inhibition becomes
stronger at higher inhibitor concentrations — a hallmark
of competitive inhibition.

**Prediction:** If phosphate is a competitive inhibitor:
- Km should increase further compared to Part 3
- Vmax should remain approximately the same

## Entering Your Data

Replace the example absorbance values with your readings from
**Column 4** of your plate.

Each row is formatted as:
`(Row label, Absorbance at 405 nm, Pathlength in cm, [pNPP] in µM)`

Do **not** change the [pNPP] concentration values.

In [ ]:
# ── Part 4: High phosphate (20 µL) — enter your Column 4 data ─
# Format: (Row label, Absorbance at 405 nm, Pathlength in cm, [pNPP] in µM)
# Replace ONLY the absorbance values.

abs405_part4 = [
    ("A", 0.00, 1.0,   0),   # 0 µM pNPP     ← replace absorbance
    ("B", 0.04, 1.0,  50),   # 50 µM pNPP    ← replace absorbance
    ("C", 0.07, 1.0, 100),   # 100 µM pNPP   ← replace absorbance
    ("D", 0.10, 1.0, 200),   # 200 µM pNPP   ← replace absorbance
    ("E", 0.13, 1.0, 400),   # 400 µM pNPP   ← replace absorbance
    ("F", 0.14, 1.0, 600),   # 600 µM pNPP   ← replace absorbance
]

activity4 = []
print("Part 4 — High Phosphate Inhibitor (20 µL)")
print(f"{'[pNPP] µM':<14} {'Absorbance (AU)':<18} {'Activity (U/min)'}")
print("-" * 50)

for row_label, abs_val, pathlength, conc_pNPP in abs405_part4:
    act = (abs_val * TOTAL_VOL_L * 1e9
           / (MOL_EXTINCTION * pathlength * ASSAY_TIME))
    act = round(act, 4)
    activity4.append((conc_pNPP, act))
    print(f"{conc_pNPP:<14} {abs_val:<18.3f} {act}")

S_data4 = np.array([d[0] for d in activity4])
V_data4 = np.array([d[1] for d in activity4])

# Scatter plot
plt.figure(figsize=(8, 5))
plt.scatter(S_data4, V_data4, color='seagreen', s=80, zorder=5)
plt.xlabel('[pNPP] (µM)', fontsize=12)
plt.ylabel('Activity (U/min)', fontsize=12)
plt.title('Part 4 — Michaelis-Menten Plot: High Phosphate (20 µL)', fontsize=13)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# ── 🔧 Update initial guesses for Part 4 ──────────────────────
vmax_init4 = 1.0    # 🔧 your estimate of Vmax
km_init4   = 200.0  # 🔧 may be larger than Part 3 Km

params4, covariance4 = curve_fit(
    michaelis_menten, S_data4, V_data4,
    p0=[vmax_init4, km_init4],
    bounds=(0, np.inf)
)
Vmax_fit4, Km_fit4 = params4

V_predicted4 = michaelis_menten(S_data4, Vmax_fit4, Km_fit4)
ss_res4 = np.sum((V_data4 - V_predicted4)**2)
ss_tot4 = np.sum((V_data4 - np.mean(V_data4))**2)
R2_mm4  = 1 - (ss_res4 / ss_tot4) if ss_tot4 > 0 else 0

print("Part 4 — High Phosphate: Michaelis-Menten Curve Fit")
print("=" * 50)
print(f"  Fitted Vmax:  {Vmax_fit4:.4f} U/min")
print(f"  Fitted Km:    {Km_fit4:.4f} µM")
print(f"  R²:           {R2_mm4:.4f}")
print(f"  Fit quality:  {'Excellent ✅' if R2_mm4>=0.99 else 'Acceptable ⚠️' if R2_mm4>=0.95 else 'Poor — check data ❌'}")

S_fit4 = np.linspace(0, max(S_data4) * 1.2, 200)
V_fit4 = michaelis_menten(S_fit4, Vmax_fit4, Km_fit4)

plt.figure(figsize=(8, 5))
plt.scatter(S_data4, V_data4, color='seagreen', s=80, zorder=5,
            label='High phosphate data')
plt.plot(S_fit4, V_fit4, color='seagreen', linewidth=2,
         label=f'MM fit: Vmax={Vmax_fit4:.3f}, Km={Km_fit4:.1f} µM')
plt.xlabel('[pNPP] (µM)', fontsize=12)
plt.ylabel('Activity (U/min)', fontsize=12)
plt.title('Part 4 — MM Curve Fit: High Phosphate (20 µL)', fontsize=13)
plt.legend(fontsize=9)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Summary — All Three Conditions

## Combined Michaelis-Menten Plot

Run the cell below to overlay all three fitted curves.

**What to look for:**
- Do the three curves approach the **same plateau** (same Vmax)?
- Do the inhibited curves approach it more slowly (higher Km)?

If yes, this is consistent with **competitive inhibition**.

In [ ]:
# Combined Michaelis-Menten plot — all three conditions
S_range = np.linspace(0, max(S_data2.max(), S_data3.max(), S_data4.max()) * 1.2, 200)

plt.figure(figsize=(10, 6))
plt.scatter(S_data2, V_data2, color='red',      s=70, zorder=5, label='No inhibitor (data)')
plt.scatter(S_data3, V_data3, color='royalblue', s=70, zorder=5, label='Low Pi — 10 µL (data)')
plt.scatter(S_data4, V_data4, color='seagreen',  s=70, zorder=5, label='High Pi — 20 µL (data)')
plt.plot(S_range, michaelis_menten(S_range, Vmax_fit2, Km_fit2), color='red',
         linewidth=2, label=f'No inh: Vmax={Vmax_fit2:.3f}, Km={Km_fit2:.1f}')
plt.plot(S_range, michaelis_menten(S_range, Vmax_fit3, Km_fit3), color='royalblue',
         linewidth=2, label=f'Low Pi: Vmax={Vmax_fit3:.3f}, Km={Km_fit3:.1f}')
plt.plot(S_range, michaelis_menten(S_range, Vmax_fit4, Km_fit4), color='seagreen',
         linewidth=2, label=f'High Pi: Vmax={Vmax_fit4:.3f}, Km={Km_fit4:.1f}')
plt.xlabel('[pNPP] (µM)', fontsize=12)
plt.ylabel('Activity (U/min)', fontsize=12)
plt.title('Combined Michaelis-Menten Curves — All Three Conditions', fontsize=13)
plt.legend(fontsize=9, loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

In [ ]:
# Results table — curve fitting
print("=" * 68)
print("          RESULTS TABLE — Michaelis-Menten Curve Fitting")
print("=" * 68)
print(f"{'Condition':<30} {'Vmax (U/min)':<16} {'Km (µM)':<12} {'R²'}")
print("-" * 68)
print(f"{'No inhibitor':<30} {Vmax_fit2:<16.4f} {Km_fit2:<12.2f} {R2_mm2:.4f}")
print(f"{'Low phosphate (10 µL)':<30} {Vmax_fit3:<16.4f} {Km_fit3:<12.2f} {R2_mm3:.4f}")
print(f"{'High phosphate (20 µL)':<30} {Vmax_fit4:<16.4f} {Km_fit4:<12.2f} {R2_mm4:.4f}")
print("=" * 68)
print()
vmax_chg = (Vmax_fit4 - Vmax_fit2) / Vmax_fit2 * 100
km_chg   = (Km_fit4   - Km_fit2)   / Km_fit2   * 100
print(f"Vmax change (no inh → high Pi): {vmax_chg:+.1f}%")
print(f"Km change   (no inh → high Pi): {km_chg:+.1f}%")
print()
if abs(vmax_chg) < 20 and km_chg > 20:
    print("✅ Pattern consistent with COMPETITIVE inhibition")
    print("   (Km increases, Vmax approximately unchanged)")
elif vmax_chg < -20 and abs(km_chg) < 20:
    print("⚠️  Pattern consistent with NON-COMPETITIVE inhibition")
    print("   (Vmax decreases, Km approximately unchanged)")
else:
    print("ℹ️  Mixed pattern — discuss with your instructor")

## Lineweaver-Burk Double-Reciprocal Plot

The **Lineweaver-Burk plot** (named after Hans Lineweaver and
Dean Burk) linearizes the Michaelis-Menten equation, making it
easier to distinguish between types of inhibition.

**How to read the plot:**
- **Y-intercept** = 1/Vmax
- **X-intercept** = −1/Km
- **Slope** = Km/Vmax

**Identifying competitive inhibition:**
Lines for different inhibitor concentrations should
**intersect on the y-axis** — same Vmax, different Km.

> ⚠️ The y-axis extends below zero so the extrapolated lines
> can cross the x-axis at −1/Km. This is the most important
> region of the plot for identifying the inhibitor type.

In [ ]:
# Lineweaver-Burk double-reciprocal plot
# Correct spelling: Lineweaver-Burk (not Burke)

# Filter out zero-concentration points (cannot divide by zero)
mask2 = S_data2 > 0;  mask3 = S_data3 > 0;  mask4 = S_data4 > 0

recip_S2 = 1/S_data2[mask2];  recip_V2 = 1/V_data2[mask2]
recip_S3 = 1/S_data3[mask3];  recip_V3 = 1/V_data3[mask3]
recip_S4 = 1/S_data4[mask4];  recip_V4 = 1/V_data4[mask4]

# Linear regression for each condition
slope2_lb, intercept2_lb, r2_lb, _, _ = linregress(recip_S2, recip_V2)
slope3_lb, intercept3_lb, r3_lb, _, _ = linregress(recip_S3, recip_V3)
slope4_lb, intercept4_lb, r4_lb, _, _ = linregress(recip_S4, recip_V4)

# Vmax = 1/y-intercept;  Km = slope × Vmax
Vmax_LB2 = 1/intercept2_lb;  Km_LB2 = slope2_lb * Vmax_LB2
Vmax_LB3 = 1/intercept3_lb;  Km_LB3 = slope3_lb * Vmax_LB3
Vmax_LB4 = 1/intercept4_lb;  Km_LB4 = slope4_lb * Vmax_LB4

# X-intercepts = -1/Km
x_int2 = -1/Km_LB2;  x_int3 = -1/Km_LB3;  x_int4 = -1/Km_LB4

# Extend lines into negative x region to show x-intercept (-1/Km)
x_min_plot = min(x_int2, x_int3, x_int4) * 1.3
x_max_plot = max(recip_S2.max(), recip_S3.max(), recip_S4.max()) * 1.1
x_fit = np.linspace(x_min_plot, x_max_plot, 300)

plt.figure(figsize=(10, 7))
plt.scatter(recip_S2, recip_V2, color='red',      s=70, zorder=5, label='No inhibitor')
plt.scatter(recip_S3, recip_V3, color='royalblue', s=70, zorder=5, label='Low Pi (10 µL)')
plt.scatter(recip_S4, recip_V4, color='seagreen',  s=70, zorder=5, label='High Pi (20 µL)')
plt.plot(x_fit, slope2_lb*x_fit + intercept2_lb, color='red',
         linestyle='--', linewidth=1.8,
         label=f'No inh: Vmax={Vmax_LB2:.3f}, Km={Km_LB2:.1f}')
plt.plot(x_fit, slope3_lb*x_fit + intercept3_lb, color='royalblue',
         linestyle='--', linewidth=1.8,
         label=f'Low Pi: Vmax={Vmax_LB3:.3f}, Km={Km_LB3:.1f}')
plt.plot(x_fit, slope4_lb*x_fit + intercept4_lb, color='seagreen',
         linestyle='--', linewidth=1.8,
         label=f'High Pi: Vmax={Vmax_LB4:.3f}, Km={Km_LB4:.1f}')

# Zero axes
plt.axhline(y=0, color='black', linewidth=0.8)
plt.axvline(x=0, color='black', linewidth=0.8)

# Mark x-intercepts (-1/Km) with an X marker
for xi, col in [(x_int2,'red'),(x_int3,'royalblue'),(x_int4,'seagreen')]:
    plt.scatter([xi], [0], color=col, s=90, marker='x', zorder=6)

plt.xlabel('1/[pNPP]  (1/µM)', fontsize=12)
plt.ylabel('1/Activity  (min/U)', fontsize=12)
plt.title('Lineweaver-Burk Double-Reciprocal Plot — All Three Conditions', fontsize=13)
plt.legend(fontsize=8, loc='upper right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print()
print("=" * 68)
print("          RESULTS TABLE — Lineweaver-Burk Analysis")
print("=" * 68)
print(f"{'Condition':<30} {'Vmax (U/min)':<16} {'Km (µM)':<12} {'r (LB)'}")
print("-" * 68)
print(f"{'No inhibitor':<30} {Vmax_LB2:<16.4f} {Km_LB2:<12.2f} {r2_lb:.4f}")
print(f"{'Low phosphate (10 µL)':<30} {Vmax_LB3:<16.4f} {Km_LB3:<12.2f} {r3_lb:.4f}")
print(f"{'High phosphate (20 µL)':<30} {Vmax_LB4:<16.4f} {Km_LB4:<12.2f} {r4_lb:.4f}")
print("=" * 68)

In [ ]:
# Side-by-side comparison of both methods
print("=" * 75)
print("       FINAL RESULTS — Curve Fitting vs. Lineweaver-Burk")
print("=" * 75)
print(f"{'Condition':<24} {'Vmax CF':>10} {'Vmax LB':>10} {'Km CF':>10} {'Km LB':>10}")
print(f"{'':24} {'(U/min)':>10} {'(U/min)':>10} {'(µM)':>10} {'(µM)':>10}")
print("-" * 75)
print(f"{'No inhibitor':<24} {Vmax_fit2:>10.4f} {Vmax_LB2:>10.4f} {Km_fit2:>10.2f} {Km_LB2:>10.2f}")
print(f"{'Low Pi (10 µL)':<24} {Vmax_fit3:>10.4f} {Vmax_LB3:>10.4f} {Km_fit3:>10.2f} {Km_LB3:>10.2f}")
print(f"{'High Pi (20 µL)':<24} {Vmax_fit4:>10.4f} {Vmax_LB4:>10.4f} {Km_fit4:>10.2f} {Km_LB4:>10.2f}")
print("=" * 75)
print()
print("CF = non-linear curve fitting (generally more accurate)")
print("LB = Lineweaver-Burk double-reciprocal plot")

# Wrap-Up and Discussion Questions

## What You Accomplished

In this notebook you:
- ✅ Converted raw absorbance readings to enzyme activity using Beer-Lambert Law
- ✅ Identified potential inhibitors from a library screen (Part 1)
- ✅ Determined Km and Vmax by non-linear Michaelis-Menten curve fitting
- ✅ Determined Km and Vmax by Lineweaver-Burk linear regression
- ✅ Compared enzyme kinetics with and without a competitive inhibitor

---

## Questions

**1. What effect does phosphate have on the kinetic properties of the enzyme?**

Look at your results table. How does Km change as you add more
phosphate? How does Vmax change? Use specific numbers.

---

**2. Is phosphate a competitive inhibitor? How do you know?**

A competitive inhibitor shows increasing Km, unchanged Vmax,
and Lineweaver-Burk lines that intersect on the y-axis.
Do your results match this pattern?

---

**3. Why does it make biological sense that phosphate would inhibit a phosphatase?**

Think about what phosphate looks like chemically, what the
enzyme does, and what the natural product of the reaction is.

---

**4. The two methods may give slightly different Vmax and Km values. Why?**

Hint: Which data points have the most influence on the slope
of a Lineweaver-Burk plot? Think about what happens when you
take the reciprocal of a small number.

---

**5. Connecting to drug discovery: What would an ideal phosphatase inhibitor drug look like, based on what you learned today?**

Consider binding affinity (Km analogy), selectivity
(which phosphatase?), and physiological concentrations.